In [1]:
import pandas as pd
import zipfile
import os
import glob
import requests
import shutil
import time
from urllib.parse import quote_plus
import pathlib
from datetime import date
import gc
import psutil
import io
import os
import io
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
#from urllib3.util.retry import Retry

from sys import getsizeof
from IPython.display import display
pasta_raiz = "//Srvl002/data/FINANCEIRO/PLANNING/INTELIGENCIA DE DADOS"
nome_do_usuario = 'samuelbarroso'


In [2]:
import yfinance as yf

pasta_raiz = "//Srvl002/data/FINANCEIRO/PLANNING/INTELIGENCIA DE DADOS"
import pandas as pd

# Lista de ativos - você pode incluir ações, ETFs e commodities
ativos = {
    'PETR4.SA': 'Petrobras',
    'WIZC3.SA': 'Wiz Co',
    'VALE3.SA': 'Vale',
    'ITUB4.SA': 'Itaú Unibanco',
    'BOVA11.SA': 'ETF BOVA11',
    'GOLD': 'Ouro',
    'CL=F': 'Petróleo WTI',
    'ZS=F': 'Soja',
    'ZW=F': 'Trigo'
}

# Período e intervalo
inicio = "2024-01-01"
fim = "2025-12-01"



for ticker, nome in ativos.items():
    df = yf.download(ticker, start=inicio, end=fim)
    df['Ativo'] = nome
    df['Ticker'] = ticker
    df['Data'] = df.index
    #df = df[['Data', 'Ativo', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume']]
    df.to_excel(pasta_raiz+"/18. Python Notebooks/"+nome+"dados_completos.xlsx")
    
"""


dados_completos = []

for ticker, nome in ativos.items():
    ticker_obj = yf.Ticker(ticker)
    info = ticker_obj.info
    info['Empresa'] = nome
    info['Ticker'] = ticker
    dados_completos.append(info)

# Criar DataFrame com todos os dados
df = pd.DataFrame(dados_completos)

# Salvar no Excel
df.to_excel(pasta_raiz+"/18. Python Notebooks/dados_completos_ativos.xlsx", index=False)


# Mostrar as primeiras linhas
print(df.tail())"""

C:\Users\samuelbarroso\AppData\Local\Temp\ipykernel_21140\159110728.py:26: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=inicio, end=fim)
[*********************100%***********************]  1 of 1 completed
C:\Users\samuelbarroso\AppData\Local\Temp\ipykernel_21140\159110728.py:26: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=inicio, end=fim)
[*********************100%***********************]  1 of 1 completed
C:\Users\samuelbarroso\AppData\Local\Temp\ipykernel_21140\159110728.py:26: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=inicio, end=fim)
[*********************100%***********************]  1 of 1 completed
C:\Users\samuelbarroso\AppData\Local\Temp\ipykernel_21140\159110728.py:26: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, star

'\n\n\ndados_completos = []\n\nfor ticker, nome in ativos.items():\n    ticker_obj = yf.Ticker(ticker)\n    info = ticker_obj.info\n    info[\'Empresa\'] = nome\n    info[\'Ticker\'] = ticker\n    dados_completos.append(info)\n\n# Criar DataFrame com todos os dados\ndf = pd.DataFrame(dados_completos)\n\n# Salvar no Excel\ndf.to_excel(pasta_raiz+"/18. Python Notebooks/dados_completos_ativos.xlsx", index=False)\n\n\n# Mostrar as primeiras linhas\nprint(df.tail())'

In [3]:
########## VARIÁVEIS GLOBAIS
pasta_raiz = "//Srvl002/data/FINANCEIRO/PLANNING/INTELIGENCIA DE DADOS"

#%whos    # mostra tamanho aproximado de cada objeto
##print(psutil.Process(os.getpid()).memory_info().rss/1e9, "GB") #Ram ocupada


In [4]:
####### FUNÇÕES GLOBAIS
def peso_script(top_n=None, decimais=3):
    """
    Exibe o tamanho (em KB) das variáveis globais com o número de casas
    decimais definido em `decimais`.

    Parâmetros
    ----------
    top_n : int ou None
        Quantas variáveis mais pesadas exibir (None = todas).
    decimais : int
        Número de casas decimais mostradas na coluna KB.
    """
    rows = []
    for nome, obj in globals().items():
        if nome.startswith('_'):
            continue
        try:
            size_kb = getsizeof(obj) / 1024      # KB
            rows.append((nome, size_kb))
        except TypeError:
            pass

    df = (pd.DataFrame(rows, columns=['Variável', 'KB'])
            .sort_values('KB', ascending=False)
            .reset_index(drop=True))

    if top_n:
        df = df.head(top_n)

    fmt = f'{{:,.{decimais}f}}'                 # monta string dinâmica
    display(df.style.format({'KB': fmt}))

In [5]:
#    GERAR DATAS FALTANTES
#
# ------------------------------------------------------------
# 1) LEITURA DO ARQUIVO EXISTENTE
# ------------------------------------------------------------
endref = pasta_raiz+"/18. Python Notebooks/2.Dados de Mercado/BACEN/IFData/Referencia.xlsx"
df = pd.read_excel(endref, dtype={"AnoMês": str})   # garante AnoMês como texto

# ------------------------------------------------------------
# 2) GERAR TODA A GRADE DE TRIMESTRES
# ------------------------------------------------------------
primeiro_ano = df["Ano"].min()
hoje         = date.today()
mes_atual    = [3, 6, 9, 12][(hoje.month - 1) // 3]   # 3, 6, 9 ou 12
ultimo_ano   = hoje.year

grade = []
meses_trim = {1: 3, 2: 6, 3: 9, 4: 12}

for ano in range(primeiro_ano, ultimo_ano + 1):
    for t, mes in meses_trim.items():
        if (ano == ultimo_ano) and (mes > mes_atual):   # não passa do trimestre corrente
            break
        grade.append({"Ano": ano,
                      "Trimestre": f"Trim{t}",
                      "Mês - Tri": f"{mes:02d}",
                      "AnoMês": f"{ano}{mes:02d}"})

df_grade = pd.DataFrame(grade)

# ------------------------------------------------------------
# 3) COMPARAR E ENCONTRAR FALTANTES
# ------------------------------------------------------------
faltantes = df_grade.loc[~df_grade["AnoMês"].isin(df["AnoMês"])]


# ------------------------------------------------------------
# 4) JUNTAR E ORDENAR
# ------------------------------------------------------------
df_completo = (
    pd.concat([df, faltantes], ignore_index=True)
      .sort_values("AnoMês")
      .reset_index(drop=True)
)

# ------------------------------------------------------------
# 5) (OPCIONAL) SALVAR RESULTADO
# ------------------------------------------------------------
df_completo.to_excel(endref, index=False)


# BACEN

### Consórcio

In [6]:
anomes = ['201801', '201802', '201803', '201804', '201805', '201806', '201807', '201808', '201809', '201810', '201811', '201812',
          '201901', '201902', '201903', '201904', '201905', '201906', '201907', '201908', '201909', '201910', '201911', '201912',
          '202001', '202002', '202003', '202004', '202005', '202006', '202007', '202008', '202009', '202010', '202011', '202012',
          '202101', '202102', '202103', '202104', '202105', '202106', '202107', '202108', '202109', '202110', '202111', '202112',
          '202201', '202202', '202203', '202204', '202205', '202206', '202207', '202208', '202209', '202210', '202211', '202212',
          '202301', '202302', '202303', '202304', '202305', '202306', '202307', '202308', '202309', '202310', '202311', '202312',
          '202401', '202402', '202403', '202404', '202405', '202406', '202407', '202408', '202409', '202410', '202411', '202412',
          '202501', '202502', '202503', '202504', '202505', '202506', '202507', '202508', '202509', '202510', '202511', '202512']


# folder path
dir_path = pasta_raiz+'/18. Python Notebooks/2.Dados de Mercado/Consórcio/Arquivos/'

# list to store files
res = []

# Iterate directory
for path in os.listdir(dir_path):
    # check if current path is a file
    if os.path.isfile(os.path.join(dir_path, path)):
        res.append(path)
#print(res)


for mes_download in anomes:
    nome_arquivo = mes_download + 'Consorcios.zip'
    if nome_arquivo in res:
        next
    else:
        url = 'https://www.bcb.gov.br/Fis/Consorcios/Port/BD/' + mes_download + 'Consorcios.zip'
        filename = pasta_raiz+'/18. Python Notebooks/2.Dados de Mercado/Consórcio/Arquivos/' + mes_download + 'Consorcios.zip'

        response = requests.get(url)
        if response.status_code != 200:
            next
        else:
            with open(filename, 'wb') as f:
                f.write(response.content)
            res.append(nome_arquivo)




# folder path
dir_path = pasta_raiz+'/18. Python Notebooks/2.Dados de Mercado/Consórcio/src/'

# list to store files
res_csv = []

# Iterate directory
for path in os.listdir(dir_path):
    # check if current path is a file
    if os.path.isfile(os.path.join(dir_path, path)):
        res_csv.append(path)
#print(res_csv)



for zip_file in res:
    with zipfile.ZipFile(pasta_raiz+"/18. Python Notebooks/2.Dados de Mercado/Consórcio/Arquivos/" + zip_file, "r") as zip_ref:
        for filename in zip_ref.namelist():
            if filename.endswith('Consolidados.csv') and filename in res_csv:
                next
            else:
                if filename.endswith('Consolidados.csv'):
                    #print(filename) 
                    zip_ref.extract(filename, path= pasta_raiz+'/18. Python Notebooks/2.Dados de Mercado/Consórcio/src/')

path = pasta_raiz+'/18. Python Notebooks/2.Dados de Mercado/Consórcio/src/*' # use your path
all_files = glob.glob(path)
all_files

list_files = []

for filename in all_files:
    df_aux = pd.read_csv(filename, index_col=None, header=0, sep=';', encoding='ISO-8859-1')
    list_files.append(df_aux)

df = pd.concat(list_files, axis=0, ignore_index=True)

df.to_csv(pasta_raiz+'/18. Python Notebooks/2.Dados de Mercado/Consórcio/arquivo_consolidado.csv', sep=';', encoding='ISO-8859-1', index=False)
df.to_parquet(pasta_raiz+'/18. Python Notebooks/2.Dados de Mercado/Consórcio/arquivo_consolidado.parquet')

with pd.ExcelWriter(pasta_raiz+'/18. Python Notebooks/2.Dados de Mercado/Consórcio/arquivo_consolidado.xlsx',
                    engine='xlsxwriter',
                    date_format='dd/mm/yyyy',      # para datas sem hora
                    datetime_format='dd/mm/yyyy') as writer: # para datas com hora 

        df.to_excel(writer, index=False)

df.to_csv("C:/Users/"+nome_do_usuario+"/Wiz Co/Ciclo & Inteligência - Documentos/Bases/11-Mercado/Consórcio/arquivo_consolidado.csv", sep=';', encoding='ISO-8859-1', index=False)
df.to_parquet("C:/Users/"+nome_do_usuario+"/Wiz Co/Ciclo & Inteligência - Documentos/Bases/11-Mercado/Consórcio/arquivo_consolidado.parquet")

with pd.ExcelWriter("C:/Users/"+nome_do_usuario+"/Wiz Co/Ciclo & Inteligência - Documentos/Bases/11-Mercado/Consórcio/arquivo_consolidado.xlsx",
                    engine='xlsxwriter',
                    date_format='dd/mm/yyyy',      # para datas sem hora
                    datetime_format='dd/mm/yyyy') as writer: # para datas com hora 

        df.to_excel(writer, index=False)


### Crédito

In [7]:
import os
import requests
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


def criar_sessao_com_retry(
    total=5,
    backoff_factor=1.3,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=("GET",),
    user_agent="IFDATA-bot 1.0",
    pool_maxsize=8,
):
    """
    Cria e retorna uma sessão requests.Session() com política de retry.
    """
    session = requests.Session()
    session.headers.update({"User-Agent": user_agent})

    retry = Retry(
        total=total,
        connect=total,
        read=total,
        backoff_factor=backoff_factor,
        status_forcelist=status_forcelist,
        allowed_methods=frozenset(allowed_methods),
        raise_on_status=False,
    )

    adapter = HTTPAdapter(max_retries=retry, pool_maxsize=pool_maxsize)
    session.mount("https://", adapter)
    session.mount("http://", adapter)

    return session


def baixar_taxa_juros_bacen(nome_do_usuario: str) -> pd.DataFrame:
    """
    Baixa a base 'TaxasJurosDiariaPorInicioPeriodo' do Bacen (Olinda),
    salva em CSV e Parquet, e retorna o DataFrame.

    - Aplica retry para erros 429/5xx.
    - Faz download em streaming para um arquivo temporário (.part)
      e só depois substitui o CSV final (escrita 'atômica').
    - Regrava o CSV com separador ';' e decimal ',' para uso direto em Excel/Power BI.
    """

    # === CONFIG ===
    url = (
        "https://olinda.bcb.gov.br/olinda/servico/taxaJuros/versao/v2/odata/"
        "TaxasJurosDiariaPorInicioPeriodo?$format=text/csv"
    )

    base_path = fr"C:\Users\{nome_do_usuario}\Wiz Co\Ciclo & Inteligência - Documentos\Bases\11-Mercado\BACEN\Crédito e Juros"
    os.makedirs(base_path, exist_ok=True)

    csv_path = fr"{base_path}\TaxaDeJuros.csv"
    parquet_path = fr"{base_path}\TaxaDeJuros.parquet"

    # === Sessão com retry ===
    session = criar_sessao_com_retry(
        total=5,
        backoff_factor=1.3,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("GET",),
        user_agent="IFDATA-bot 1.0",
        pool_maxsize=8,
    )

    # === Download em streaming ===
    with session.get(url, stream=True, timeout=(10, 600)) as r:
        r.raise_for_status()

        ctype = (r.headers.get("Content-Type") or "").lower()
        if "text/csv" not in ctype:
            print(f"Aviso: Content-Type inesperado: {ctype}")

        tmp_path = csv_path + ".part"
        with open(tmp_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):  # 1 MB
                if chunk:
                    f.write(chunk)

        # Move atômico: só aqui substitui o arquivo final
        os.replace(tmp_path, csv_path)

    # === Ler no pandas e salvar em CSV formatado + Parquet ===
    df = pd.read_csv(csv_path, sep=",")
    df.to_csv(csv_path, sep=";", decimal=",", index=False, encoding="ISO-8859-1")
    df.to_parquet(parquet_path)

    return df

df_taxa = baixar_taxa_juros_bacen(nome_do_usuario)


In [8]:
# dataInicial e dataFinal formato: 'dd/mm/yyyy'
def consulta_bacen(codigo_bcb, dataInicial, dataFinal, column_name):
    if type(codigo_bcb) == list:
        url = 'https://api.bcb.gov.br/dados/serie/bcdata.sgs.{}/dados?formato=csv&dataInicial={}&dataFinal={}'.format(codigo_bcb[0], dataInicial, dataFinal)
        i = 1
        df = pd.read_csv(url, sep=';')
        tam_cod = len(codigo_bcb)
        if tam_cod != len(column_name):
            print('O tamanho da lista do código do bacen não é igual a quantidade de nome das colunas!')
            return
        while i < tam_cod:
            url = 'https://api.bcb.gov.br/dados/serie/bcdata.sgs.{}/dados?formato=csv&dataInicial={}&dataFinal={}'.format(codigo_bcb[i], dataInicial, dataFinal)
            df_aux = pd.read_csv(url, sep=';')
            df = df.merge(df_aux, how='outer', on='data', suffixes=(i-1, i))
            i += 1
        df_columns = ['Data'] + column_name
        df.columns = df_columns
        return df
    else:
        url = 'https://api.bcb.gov.br/dados/serie/bcdata.sgs.{}/dados?formato=csv&dataInicial={}&dataFinal={}'.format(codigo_bcb, dataInicial, dataFinal)
        df = pd.read_csv(url, sep=';')
        df_columns = ['Data'].append(column_name)
        df.columns = df_columns
        return df
    

dados_crédito = consulta_bacen([20666, 20670, 20671, 432, 20667], '01/01/2018', '31/12/2026', 
                                 ['Crédito PF não consignado - R$ (milhões)', 
                                  'Crédito PF consignado INSS - R$ (milhões)', 
                                  'Crédito PF consignado total - R$ (milhões)',
                                  'Taxa de juros - Meta Selic definida pelo Copom',
                                  '20667 - Concessões de crédito com recursos livres - Pessoas físicas - Crédito pessoal não consignado vinculado à composição de dívidas - R$ (milhões)'])

# projeção selic
data = pd.read_csv('https://olinda.bcb.gov.br/olinda/servico/Expectativas/versao/v1/odata/ExpectativasMercadoAnuais?$format=text/csv&$select=Indicador,Data,DataReferencia,Media,Mediana,DesvioPadrao,Minimo,Maximo,numeroRespondentes,baseCalculo')
df = data[(data['Indicador'] == 'Selic') & (data['Data'] == data['Data'].max()) & (data['baseCalculo'] == 1)]
df['Anomes'] = df['DataReferencia'].astype('str')  + '12'
df['DataPrev'] = df['DataReferencia'].astype('str') + '-12-01'



dados_crédito['Data'] = pd.to_datetime(dados_crédito['Data'], dayfirst=True)

dados_crédito['Projecao Selic Mediana'] = None
dados_crédito['Projecao Selic Minima'] = None
dados_crédito['Projecao Selic Maxima'] = None
new_row = {'Data': dados_crédito['Data'].max().replace(day=1), 
           'Projecao Selic Mediana': dados_crédito[dados_crédito['Data'] == dados_crédito['Data'].max()]['Taxa de juros - Meta Selic definida pelo Copom'].values[0],
           'Projecao Selic Minima': dados_crédito[dados_crédito['Data'] == dados_crédito['Data'].max()]['Taxa de juros - Meta Selic definida pelo Copom'].values[0],
           'Projecao Selic Maxima': dados_crédito[dados_crédito['Data'] == dados_crédito['Data'].max()]['Taxa de juros - Meta Selic definida pelo Copom'].values[0]}
dados_crédito.loc[len(dados_crédito)] = new_row



df = df[['DataPrev', 'Mediana', 'Minimo', 'Maximo']]
df.columns = ['Data', 'Projecao Selic Mediana', 'Projecao Selic Minima', 'Projecao Selic Maxima']
df['Data'] = pd.to_datetime(df['Data'])


df_final = pd.concat([dados_crédito, df])

df_final.to_csv("C:/Users/"+nome_do_usuario+"/Wiz Co/Ciclo & Inteligência - Documentos/Bases/11-Mercado/BACEN/Crédito e Juros/base_mercado_bacen.csv", sep=';', decimal=',', index=False, encoding='ISO-8859-1')
df_final.to_parquet("C:/Users/"+nome_do_usuario+"/Wiz Co/Ciclo & Inteligência - Documentos/Bases/11-Mercado/BACEN/Crédito e Juros/base_mercado_bacen.parquet")

with pd.ExcelWriter("C:/Users/"+nome_do_usuario+"/Wiz Co/Ciclo & Inteligência - Documentos/Bases/11-Mercado/BACEN/Crédito e Juros/base_mercado_bacen.xlsx",
                    engine='xlsxwriter',
                    date_format='dd/mm/yyyy',      # para datas sem hora
                    datetime_format='dd/mm/yyyy') as writer: # para datas com hora 

        df_final.to_excel(writer, index=False)





C:\Users\samuelbarroso\AppData\Local\Temp\ipykernel_21140\499677906.py:37: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Anomes'] = df['DataReferencia'].astype('str')  + '12'
C:\Users\samuelbarroso\AppData\Local\Temp\ipykernel_21140\499677906.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['DataPrev'] = df['DataReferencia'].astype('str') + '-12-01'


### IFDATA

In [9]:
def criar_pasta(pasta_raiz_, pasta_tipo_,pasta_relat_,tipo):
    if tipo ==1:
        dest_dir_ = pathlib.Path(
            pasta_raiz_,
            "18. Python Notebooks",
            "2.Dados de Mercado",
            "BACEN",
            "IFData",
            str(pasta_tipo_),
            str(pasta_relat_)
        )
        dest_dir_.mkdir(parents=True, exist_ok=True)
    elif tipo ==2:
        dest_dir_ = pathlib.Path(
            pasta_raiz_,
            "18. Python Notebooks",
            "2.Dados de Mercado",
            "BACEN",
            "IFData",
            "Consolidado",
            str(pasta_tipo_),
            str(pasta_relat_)
        )
        dest_dir_.mkdir(parents=True, exist_ok=True)
    else:
        dest_dir_ = pathlib.Path(
            "C:/Users/"+nome_do_usuario+"/Wiz Co/Ciclo & Inteligência - Documentos/Bases",
            "11-Mercado",
            "BACEN",
            "IFDATA",
            "Consolidado",
            str(pasta_tipo_),
            str(pasta_relat_)
        )
        dest_dir_.mkdir(parents=True, exist_ok=True)
    return dest_dir_

def fetch_df(am, tipo, relat, tries=3, pause=0.3):
    """Busca TUDO (todas as páginas) para um dado AnoMes/Tipo/Relatorio."""
    # evita '1.0' vindo de Excel
    relat_str = str(relat).rstrip(".0")
    url = BASE.format(am=am, tipo=tipo, relat=quote_plus(relat_str))

    for n in range(tries):
        try:
            frames = []
            while url:
                r = SESSION.get(url, timeout=60)
                if r.status_code == 200:
                    j = r.json()
                    frames.append(pd.DataFrame(j.get("value", [])))
                    # segue para a próxima página, se existir
                    url = j.get("@odata.nextLink") or j.get("odata.nextLink")
                elif r.status_code == 429:
                    time.sleep((2**n) * pause)  # backoff
                else:
                    print(f"⚠️ {r.status_code} em {url}")
                    return pd.DataFrame()
            return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
        except requests.RequestException as e:
            print(f"⚠️ erro de rede: {e}; tentativa {n+1}/{tries}")
            time.sleep((2**n) * pause)

    return pd.DataFrame()

In [10]:

endref = pasta_raiz+"/18. Python Notebooks/2.Dados de Mercado/BACEN/IFData/Referencia.xlsx"
endrelat = pasta_raiz+'/18. Python Notebooks/2.Dados de Mercado/BACEN/IFData/Relatorios.xlsx'

df_ref = pd.read_excel(endref)
df_relat = pd.read_excel(endrelat)

listadata = df_ref['AnoMês'].to_list()
Relat_list = [1,2,3]


SESSION          = requests.Session()        # ➊  reaproveita conexão
SESSION.headers.update({"User-Agent": "IFDATA-bot 1.0"})

BASE = ("https://olinda.bcb.gov.br/olinda/servico/IFDATA/versao/v1/odata/"
        "IfDataValores(AnoMes=@AnoMes,TipoInstituicao=@TipoInstituicao,"
        "Relatorio=@Relatorio)"
        "?@AnoMes={am}&@TipoInstituicao={tipo}&@Relatorio='{relat}'"
        "&$format=json")           # ➋  aumenta inteiro do lote

teste = 0
SUBDIRS  = False 
padrao   = "**/*.parquet" if SUBDIRS else "*.parquet"
# -------------------------------------------------------------------------

for tipo in Relat_list:                                          # 1,2,3
    pasta_tipo = (df_relat.loc[df_relat["Filtro"] == tipo, "NomePasta"].iloc[0])                                  # <- pega string    
    relats = df_relat.loc[df_relat["Filtro"] == tipo, "NumRelat"]
    print(f"✅ Nome: {pasta_tipo}")
    
    for relat in relats:
        
        dfs = []                                                    # zera aqui
        pasta_relat = (df_relat.loc[(df_relat["Filtro"] == tipo) & (df_relat["NumRelat"] == relat),"NomeRelatorio2"].iloc[0])
        dest_dir = criar_pasta(pasta_raiz, pasta_tipo,pasta_relat,1)
        print(f"✅ Relatorio: {pasta_relat}")

        for am in listadata:                                     # 202401, …
            
            # 1. monta o path final
            file_path = dest_dir / f"{am}.parquet"
            teste=1
            # 2. verifica existência
            if file_path.exists():
                print(f"✅ Já existe, pulando: {file_path.name}")
                continue      # ---> não baixa, vai pro próximo código
            df = fetch_df(am, tipo, relat)
            if df.empty:
                continue
            teste=1
            # (a) salva cada mês isolado
            df.to_parquet(dest_dir / f"{am}.parquet", index=False)
        if teste ==1:
            print("Salvando")
            dest_dir_ = criar_pasta(pasta_raiz, pasta_tipo,pasta_relat,2)
            arquivos = sorted(dest_dir.glob(padrao))
            dfs = [pd.read_parquet(arq) for arq in arquivos]
            df_final = pd.concat(dfs, ignore_index=True)  # colunas devem coincidir
            df_final.to_parquet(dest_dir_ / "all_periods.parquet", index=False)
            dest_dir_ = criar_pasta(pasta_raiz, pasta_tipo,pasta_relat,0)
            df_final.to_parquet(dest_dir_ / "all_periods.parquet", index=False)
            teste = 0
            

del dfs, dest_dir, dest_dir_, df, teste, pasta_relat, pasta_tipo, SUBDIRS, SESSION, endref, endrelat, relats,df_ref, df_relat
gc.collect()
# -------------------------------------------------------------------------
#df_final = pd.concat(dfs, ignore_index=True)                     # ➍  concatena só 1x
print(df_final.shape)

✅ Nome: ConglomeradosPrudenciais
✅ Relatorio: Resumo
✅ Já existe, pulando: 202503.parquet
✅ Já existe, pulando: 202506.parquet
Salvando
✅ Relatorio: Ativo
✅ Já existe, pulando: 202503.parquet
✅ Já existe, pulando: 202506.parquet
Salvando
✅ Relatorio: Passivo
✅ Já existe, pulando: 202503.parquet
✅ Já existe, pulando: 202506.parquet
Salvando
✅ Relatorio: DemonstracoesFinanceiras
✅ Já existe, pulando: 202503.parquet
✅ Já existe, pulando: 202506.parquet
Salvando
✅ Relatorio: InformacoesdeCapital
✅ Já existe, pulando: 202503.parquet
✅ Já existe, pulando: 202506.parquet
Salvando
✅ Relatorio: Segmentacao
✅ Já existe, pulando: 202503.parquet
✅ Já existe, pulando: 202506.parquet
Salvando
✅ Relatorio: CartAtiv_Indexador
✅ Já existe, pulando: 202503.parquet
✅ Já existe, pulando: 202506.parquet
Salvando
✅ Relatorio: CartAtiv_Regiao
✅ Já existe, pulando: 202503.parquet
✅ Já existe, pulando: 202506.parquet
Salvando
✅ Relatorio: CartAtiv_QtdCliente
✅ Já existe, pulando: 202503.parquet
✅ Já existe, pu

# IBGE

# Seguros

In [11]:
url = 'https://www2.susep.gov.br/redarq.asp?arq=BaseCompleta%2ezip'
filename = pasta_raiz+'/18. Python Notebooks/2.Dados de Mercado/Seguros/src/BaseCompleta.zip'

response = requests.get(url, verify=False)

with open(filename, 'wb') as f:
    f.write(response.content)

#print(f'O arquivo {filename} foi baixado com sucesso.')

base_path = fr"C:\Users\{nome_do_usuario}\Wiz Co\Ciclo & Inteligência - Documentos\Bases\11-Mercado\Seguros"
parquet_path  = fr"{base_path}\Seguros.parquet"
parquet_path_UF  = fr"{base_path}\Seguros_UF.parquet"

with zipfile.ZipFile(pasta_raiz+"/18. Python Notebooks/2.Dados de Mercado/Seguros/src/BaseCompleta.zip", "r") as zip_ref:
    zip_ref.extract("Ses_seguros.csv", path=pasta_raiz+'/18. Python Notebooks/2.Dados de Mercado/Seguros/src')
    zip_ref.extract("Ses_grupos_economicos.csv", path=pasta_raiz+'/18. Python Notebooks/2.Dados de Mercado/Seguros/src')
    zip_ref.extract("SES_UF2.csv", path=pasta_raiz+'/18. Python Notebooks/2.Dados de Mercado/Seguros/src')
    #zip_ref.extract("Ses_ramos.csv", path=pasta_raiz+'/18. Python Notebooks/2.Dados de Mercado/Seguros/src')

df_seguros = pd.read_csv(pasta_raiz+"/18. Python Notebooks/2.Dados de Mercado/Seguros/src/Ses_seguros.csv", on_bad_lines='skip', sep=';', decimal=',')
df_grupos = pd.read_csv(pasta_raiz+"/18. Python Notebooks/2.Dados de Mercado/Seguros/src/Ses_grupos_economicos.csv", on_bad_lines='skip', sep=';', decimal=',', encoding="ISO-8859-1")
df_ramos = pd.read_csv(pasta_raiz+"/18. Python Notebooks/2.Dados de Mercado/Seguros/src/Ses_ramos.csv", on_bad_lines='skip', sep=';', decimal=',', encoding="ISO-8859-1")
df_UFs = pd.read_csv(pasta_raiz+"/18. Python Notebooks/2.Dados de Mercado/Seguros/src/SES_UF2.csv", on_bad_lines='skip', sep=';', decimal=',', encoding="ISO-8859-1")


c:\Users\samuelbarroso\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1103: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www2.susep.gov.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\samuelbarroso\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1103: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www2.susep.gov.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [12]:
df_UFs = df_UFs[
    (df_UFs['premio_dir'].notna()) &    # remove valores NaN
    (df_UFs['premio_dir'] != 0) &       # remove valores zerados
    (df_UFs['premio_dir'] != '')        # remove strings vazias (caso existam)
]
df_UFs = (
    df_UFs
    .groupby(['damesano', 'coenti', 'ramos', 'UF'], as_index=False)
    .agg(
        premio_dir2=('premio_dir', 'sum')
    )
)
agregado = (
    df_UFs
    .groupby(['damesano', 'coenti', 'ramos'], as_index=False)
    .agg(
        total_valor=('premio_dir2', 'sum'),
        qtd_linhas=('premio_dir2', 'count')
    )
)

df_UFs = pd.merge(
    df_UFs,
    agregado,
    on=['damesano', 'coenti', 'ramos'],  # chaves de junção
    how='left',   # ou 'inner', 'left', 'right' conforme a necessidade
    suffixes=('_base1', '_base2')  # para diferenciar colunas iguais
)
df_UFs['percentual']=df_UFs['premio_dir2'] /df_UFs['total_valor']
df_UFs = df_UFs.drop(columns={"premio_dir2","total_valor"})
df_UFs = df_UFs.rename(columns={
            "ramos": "coramo"
        })

df_seguros = df_seguros.fillna(0)
df_seguros['coramo'] = df_seguros['coramo'].astype('int64')
agregado = (
    df_seguros
    .groupby(['damesano','cogrupo', 'coenti', 'coramo'], as_index=False)
    .agg(
        premio_de_seguros=('premio_de_seguros', 'sum')
    )
)
agregado = agregado[
    (agregado['premio_de_seguros'].notna()) &    # remove valores NaN
    (agregado['premio_de_seguros'] != 0) &       # remove valores zerados
    (agregado['premio_de_seguros'] != '')        # remove strings vazias (caso existam)
]
df_UF3 = pd.merge(
    agregado,
    df_UFs,
    on=['damesano', 'coenti', 'coramo'],  # chaves de junção
    how='left',   # ou 'inner', 'left', 'right' conforme a necessidade
    suffixes=('_base1', '_base2')  # para diferenciar colunas iguais
)

df_UF3['percentual'] = df_UF3['percentual'].replace('', 1)
df_UF3['percentual'] = df_UF3['percentual'].fillna(1)

df_UF3['qtd_linhas'] = df_UF3['qtd_linhas'].replace('', 1)
df_UF3['qtd_linhas'] = df_UF3['qtd_linhas'].fillna(1)


df_UF3['premio'] = df_UF3['premio_de_seguros'].mul(df_UF3['percentual'], axis=0)
# Identificar os grupos que têm pelo menos um NaN em UF
grupos_invalidos = (
    df_UF3[df_UF3['UF'].isna()]
    .groupby(['damesano','cogrupo','coenti','coramo'])
    .size()
    .reset_index()[['damesano','cogrupo','coenti','coramo']]
)

# Remover esses grupos do DataFrame original (anti-join)
df_UF3 = df_UF3.merge(grupos_invalidos, 
                       on=['damesano','cogrupo','coenti','coramo'], 
                       how='left', indicator=True)
df_UF3 = df_UF3[df_UF3['_merge'] == 'left_only'].drop(columns=['_merge'])

colunas_melt = ['premio_de_seguros','qtd_linhas','percentual','ponderada']
colunas_melt = ['premio']
df_UF3 = df_UF3.drop(columns={'premio_de_seguros','qtd_linhas','percentual'})
df_UF3 = df_UF3.melt(
    id_vars=['damesano','coenti','cogrupo','coramo','UF'],           # colunas que você quer manter fixas
    value_vars=colunas_melt, # colunas que você quer transformar em linhas
    var_name='Tipo',           # nome da nova coluna que vai indicar o tipo
    value_name='Valor'         # nome da nova coluna com os valores
)
df_UF3 = df_UF3[
    (df_UF3['Valor'].notna()) &    # remove valores NaN
    (df_UF3['Valor'] != 0) &       # remove valores zerados
    (df_UF3['Valor'] != '')        # remove strings vazias (caso existam)
]
df_UF3.to_parquet(pasta_raiz+'/18. Python Notebooks/2.Dados de Mercado/Seguros/Seguros_UF.parquet')
df_UF3.to_parquet(parquet_path_UF)


In [13]:

del agregado,df_UF3, grupos_invalidos, df_UFs
gc.collect()

df_grupos['noenti'] = df_grupos['noenti'].str.strip()
df_grupos.loc[df_grupos['coenti'] == 5118,'cogrupo'] = 19
df_grupos.loc[df_grupos['coenti'] == 5118,'nogrupo'] = 'SUL AMERICA'
df_grupos.dropna(inplace=True)
df_grupos['damesano'] = df_grupos['damesano'].astype('int64')
df_grupos['cogrupo'] = df_grupos['cogrupo'].astype('int64')



df_final = df_seguros.merge(df_grupos, how='left', on=['damesano', 'coenti'], suffixes=('', '_y'))
df_final = df_final.fillna(0)
df_final = df_final.drop(columns='cogrupo_y')


df_final = df_final.merge(df_ramos, how='left', on='coramo')
df_final = df_final.fillna(0)
df_final["noenti"] = df_final["noenti"].astype("string")
df_final["nogrupo"] = df_final["nogrupo"].astype("string")
df_final["noramo"] = df_final["noramo"].astype("string")
df_final["ramo_wiz"] = df_final["ramo_wiz"].astype("string")
df_final.to_parquet(pasta_raiz+'/18. Python Notebooks/2.Dados de Mercado/Seguros/mercado_seguros.parquet')
df_final.to_parquet(parquet_path)

df_excel = df_final[df_final['damesano'] >= 202300]
if df_excel[df_excel['noramo'] == 0].shape[0] != 0:
    print('HÁ NOVOS RAMOS A SEREM INCLUÍDOS NO ARQUIVO \'Ses_ramos.csv\' na coluna \'noramo\':')
    print('--------------------------------------------------------------')
    print(df_excel[df_excel['noramo'] == 0])

if df_excel[df_excel['ramo_wiz'] == 0].shape[0] != 0:
    print('--------------------------------------------------------------')
    print('--------------------------------------------------------------')
    print('HÁ NOVOS RAMOS A SEREM INCLUÍDOS NO ARQUIVO \'Ses_ramos.csv\' na coluna \'ramo_wiz\':')
    print('--------------------------------------------------------------')
    print(df_excel[df_excel['noramo'] == 0])


df_excel.to_excel(pasta_raiz+'/18. Python Notebooks/2.Dados de Mercado/Seguros/mercado_seguros.xlsx', index=False)
df_excel.to_excel(fr"{base_path}\Seguros.xlsx", index=False)



# Outro